# Week 13 - Case Study: Real-World Dataset Analysis

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 13 - Case Study**

---

## Session topics (from the course outline)

- End-to-end analysis of a real dataset
- Combining cleaning, manipulation, visualization and statistics
- Presenting findings

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 3** | Apply Python libraries (NumPy, Pandas) to clean, manipulate and process datasets |
| **CLO 4** | Use Matplotlib and Seaborn to create charts for effective presentation |
| **CLO 5** | Perform exploratory data analysis (EDA) to uncover insights |
| **CLO 8** | Evaluate the effectiveness of different visualization methods |
| **CLO 9** | Design interactive dashboards or visual reports to present data findings |

## Reading

| Topic | Source |
|---|---|
| Data analysis examples | McKinney §13.1–13.5 |
| Case studies | Pfaffelhuber §10 |

## What this week is

Twelve weeks of technique, applied once, to one dataset, from a question to an answer somebody
could act on.

Nothing in this notebook is new. Everything in it is from Weeks 1 to 12: loading and auditing,
cleaning, `groupby`, merging, resampling, plotting, and a confidence interval. What is new is
the **order** — and the discovery that the order matters more than any individual technique.

This session is also the template for your final project. The structure of this notebook is the
structure your project report should have, and section 2.8 is the form your findings should
take.

### A warning about the answer

The case study has a twist, and it is the reason this session exists. The first answer the data
appear to give is a **15% increase**. By the end of section 2.7 that number has gone to zero, and
what remains is a finding of a different kind. Getting there takes three separate pieces of
domain knowledge, none of which is in the numbers.

That is not a contrived teaching example. It is the ordinary condition of applied work, and the
habit that protects you is the one this session is built to install: **before believing a
result, ask what else changed at the same time.**

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 — Theory and concepts

---

## 1.1 The question comes first

The most common way to waste a week is to open a dataset and start exploring. Exploration with
no question produces charts with no audience.

A usable analysis question has four properties.

| Property | Bad | Better |
|---|---|---|
| **Specific** | "Analyse the sales data" | "Did the loyalty programme raise sales?" |
| **Measurable** | "Is the café doing well?" | "What is mean daily revenue per branch?" |
| **Bounded in time** | "Are we growing?" | "Did units sold per day rise between Q3 and Q4 2025?" |
| **Actionable** | "Which items are popular?" | "Which items should the smallest branch stop carrying?" |

Write the question down before loading the file, and write down **what answer would change what
somebody does**. If no answer changes anything, the analysis is optional and you should say so
rather than doing it.

---

## 1.2 The workflow

```
   QUESTION  ->  ACQUIRE  ->  AUDIT  ->  CLEAN  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
                                ^                      |
                                +----------------------+
                                  every loop back is normal
```

Drawn as a line it is a lie. In practice exploration keeps sending you back to cleaning,
because a chart is the fastest way to discover that a column has two spellings of the same
value. Budget for the loop; the usual split is that cleaning and auditing take more time than
everything else combined.

The stage most often skipped is **audit**, and it is the one that costs the most when skipped.
Auditing is deliberately looking at what the data claim to be — shape, types, missingness,
duplicates, ranges, categories — **before** forming any opinion about what they say.

---

## 1.3 The data-quality audit

Six questions, asked of every dataset, in this order:

| # | Question | How it shows up |
|---|---|---|
| 1 | How big, and what types? | `.shape`, `.dtypes` |
| 2 | What is missing, and is it missing at random? | `.isna().sum()`, missingness by group |
| 3 | Are there duplicates? On which key? | `.duplicated(subset=...)` |
| 4 | Are the categories what they claim? | `.value_counts()` on every text column |
| 5 | Are the numbers possible? | `.describe()`, then check min and max against reality |
| 6 | Does the time axis have gaps or spikes? | plot the daily count |

Question 4 catches the most errors in practice. `value_counts()` on a text column takes two
seconds and routinely reveals that one category has three spellings, which quietly splits a
group in every subsequent `groupby`.

Question 6 is the one people forget. A gap in a daily series usually means an outage, a
closure, or a failed export, and averaging across it produces a number that describes nothing.

---

## 1.4 Confounding, and why before-and-after is the weakest design

The most common applied comparison is: something changed on date $T$; compare the period before
with the period after.

It is the weakest design there is, because *everything else also changed at $T$* — the season,
the prices, the weather, the term dates, the staff. A before-and-after difference measures the
intervention **plus** everything else that moved.

Formally, a confounder is a variable that affects both the treatment and the outcome. If prices
rose two weeks before a loyalty programme launched, then "revenue after minus revenue before"
contains the price rise, and no amount of statistical care applied to that number will remove it.

Three defences, in increasing order of strength:

1. **Measure the outcome in units the confounder does not touch.** If the confounder is price,
   measure volume.
2. **Find a control** — a comparable unit that did *not* receive the intervention but was
   exposed to everything else.
3. **Randomise** — assign the intervention at random. Rarely available after the fact, which is
   why the design has to be decided before the launch, not after.

---

## 1.5 Difference-in-differences

With a control group, the estimator is:

$$\widehat{\text{effect}} \;=\; \underbrace{(\bar{y}^{\,T}_{\text{after}} - \bar{y}^{\,T}_{\text{before}})}_{\text{treated change}} \;-\; \underbrace{(\bar{y}^{\,C}_{\text{after}} - \bar{y}^{\,C}_{\text{before}})}_{\text{control change}}$$

The control's change estimates what would have happened to the treated group anyway, so
subtracting it removes everything that affected both.

When the outcome is naturally multiplicative — sales, traffic, counts — take logs first. The
difference of differences is then approximately the **percentage** effect:

$$\text{percentage effect} \;=\; \exp\!\left(\widehat{\text{effect}}_{\log}\right) - 1$$

**The assumption you are making** is *parallel trends*: without the intervention, treated and
control would have moved together. It is not testable for the period after the intervention,
which is the whole difficulty. What you can do is check that they moved together **before** it —
and section 2.7 does exactly that, because an estimator whose assumption you have not looked at
is a number with no standing.

---

## 1.6 Reproducibility

An analysis nobody can re-run is an opinion. Four habits make it re-runnable:

- **Fix the seed.** Every notebook in this course starts with `RANDOM_SEED = 42`.
- **Record versions.** The setup cell prints `numpy`, `pandas` and `seaborn` versions for this
  reason.
- **Never edit the raw data.** Load it, and produce a cleaned copy. The cleaning must be code,
  not a sequence of manual corrections, so that it can be re-applied when the data are refreshed.
- **Restart and Run All before you submit.** A notebook that only works in the order you
  happened to click is not reproducible, and out-of-order execution is the single most common
  reason a submitted notebook fails on the marker's machine.

---

## 1.7 What a finding looks like when it is written down

A finding is not a chart. It is a sentence with five parts:

1. **What** was measured, in the units of the problem.
2. **How much** — the estimate.
3. **How certain** — an interval, or an explicit statement that it cannot be pinned down.
4. **Compared with what** — the baseline or the control.
5. **So what** — the decision it bears on.

Compare:

> *"Revenue increased after the loyalty programme launched."*

with

> *"Units sold per day at the three loyalty branches moved −0.5% relative to the control branch
> after launch (95% CI −10% to +9%). The data cannot distinguish that from no effect in either
> direction, so the programme should not be extended on this evidence; a staggered rollout
> across branches would settle it in one quarter."*

The second is the same analysis, written so a manager can act on it. Producing that sentence is
the assessable skill of this week, and of the final project.

---

---

# Part 2 — The case study

---

## 2.1 The brief

> **Bopha Coffee** runs four branches in Phnom Penh. On **1 October 2025** it launched a
> loyalty programme at three of them — Riverside, Toul Kork and Sen Sok. The Airport branch
> did not get it, because its till software had not been upgraded.
>
> It is now April 2026. The owner wants to know whether to extend the programme to the Airport
> branch and renew the contract with the loyalty provider, which costs about **$450 a month**.
>
> You have been given a transaction-level export from the till system covering
> 1 January 2025 to 31 March 2026.

**The questions, written down before the data are opened.**

| # | Question | What the answer would change |
|---|---|---|
| Q1 | How do the four branches compare on daily revenue and volume? | Where to put effort |
| Q2 | What does the weekly pattern look like, and does it differ by branch? | Staffing |
| Q3 | **Did the loyalty programme increase sales?** | Whether to renew a $450/month contract |

Q3 is the one the owner is paying for. Q1 and Q2 exist because you cannot answer Q3 without
first knowing what normal looks like.

---

## 2.2 Acquire and audit

The generator below stands in for the till export. In a real project this would be
`pd.read_csv("bopha_export.csv")` and you would not be able to see how the data were made —
which is the situation section 2.6 is about.

In [ ]:
BRANCHES = {
    "Riverside": {"base": 40.0, "district": "Daun Penh",   "seats": 48, "opened": "2021-06-14"},
    "Toul Kork": {"base": 32.0, "district": "Toul Kork",   "seats": 36, "opened": "2022-02-01"},
    "Sen Sok":   {"base": 26.0, "district": "Sen Sok",     "seats": 30, "opened": "2023-09-11"},
    "Airport":   {"base": 22.0, "district": "Por Senchey", "seats": 18, "opened": "2024-05-20"},
}

MENU = {"Iced coffee": 1.75, "Hot coffee": 1.50, "Iced latte": 2.25, "Tea": 1.25,
        "Pastry": 1.60, "Sandwich": 3.50, "Bottled water": 0.75}
ITEM_SHARE = [0.30, 0.16, 0.14, 0.10, 0.15, 0.08, 0.07]

START = pd.Timestamp("2025-01-01")
END = pd.Timestamp("2026-03-31")
PRICE_RISE = pd.Timestamp("2025-09-15")
LOYALTY_LAUNCH = pd.Timestamp("2025-10-01")
RENOVATION_START = pd.Timestamp("2025-08-10")
RENOVATION_END = pd.Timestamp("2025-08-31")
LOYALTY_BRANCHES = ["Riverside", "Toul Kork", "Sen Sok"]
CONTROL_BRANCH = ["Airport"]

# University terms. The Toul Kork branch is beside a campus.
TERM_WINDOWS = [("2025-01-06", "2025-05-30"), ("2025-08-16", "2025-12-20"),
                ("2026-01-05", "2026-05-29")]

HOURS = np.arange(6, 21)
HOUR_SHARE = np.array([0.03, 0.09, 0.13, 0.10, 0.07, 0.06, 0.08, 0.07,
                       0.06, 0.07, 0.06, 0.05, 0.05, 0.05, 0.03])


def in_term(day) -> bool:
    """Is this day inside a university term?"""
    return any(pd.Timestamp(a) <= day <= pd.Timestamp(b) for a, b in TERM_WINDOWS)


def expected_transactions(branch: str, day) -> float:
    """Expected number of till transactions for one branch on one day."""
    count = BRANCHES[branch]["base"]

    weekday = day.dayofweek
    if branch == "Riverside":                       # riverside promenade: busy at weekends
        count *= [0.95, 0.95, 1.00, 1.00, 1.10, 1.25, 1.15][weekday]
    elif branch in ("Toul Kork", "Sen Sok"):        # offices and campus: quiet at weekends
        count *= [1.10, 1.10, 1.10, 1.08, 1.05, 0.70, 0.55][weekday]

    if branch == "Toul Kork" and in_term(day):      # term time
        count *= 1.18
    count *= 1.0 + 0.00025 * (day - START).days     # slow underlying growth
    if branch in LOYALTY_BRANCHES and day >= LOYALTY_LAUNCH:
        count *= 1.03
    if branch in CONTROL_BRANCH and RENOVATION_START <= day <= RENOVATION_END:
        count *= 0.12                               # refit: one till open
    return count

In [ ]:
def build_raw_extract() -> pd.DataFrame:
    """Stand-in for the till export, complete with the faults a real export has."""
    days = pd.date_range(START, END, freq="D")
    items = list(MENU)
    pieces = []

    for branch in BRANCHES:
        expected = np.array([expected_transactions(branch, day) for day in days])
        counts = rng.poisson(expected)
        day_of = np.repeat(days.to_numpy(), counts)
        n = day_of.size

        chosen = rng.choice(items, size=n, p=ITEM_SHARE)
        price = np.array([MENU[name] for name in chosen])
        price = np.where(day_of >= PRICE_RISE.to_numpy(), price * 1.10, price)

        if branch in LOYALTY_BRANCHES:
            member = (day_of >= LOYALTY_LAUNCH.to_numpy()) & (rng.random(n) < 0.38)
        else:
            member = np.zeros(n, dtype=bool)

        pieces.append(pd.DataFrame({
            "branch": branch,
            "date": day_of,
            "hour": rng.choice(HOURS, size=n, p=HOUR_SHARE),
            "item": chosen,
            "quantity": rng.choice([1, 1, 1, 1, 2, 2, 3], size=n),
            "unit_price_usd": price.round(2),
            "payment": rng.choice(["Cash", "QR", "Card"], size=n, p=[0.45, 0.42, 0.13]),
            "loyalty_member": member,
        }))

    raw = pd.concat(pieces, ignore_index=True)
    raw = raw.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    raw.insert(0, "transaction_id", [f"T{i:07d}" for i in range(1, len(raw) + 1)])
    n = len(raw)

    # (a) refunds are recorded as negative quantities on the same till
    refunds = rng.choice(n, size=int(0.008 * n), replace=False)
    raw.loc[refunds, "quantity"] = -raw.loc[refunds, "quantity"]

    # (b) staff type the branch name by hand at the start of each shift
    variants = {
        "Riverside": ["riverside", "Riverside "],
        "Toul Kork": ["toul kork", "Toul  Kork", "TOUL KORK"],
        "Sen Sok": ["sen sok", "Sen sok"],
        "Airport": ["airport", "AIRPORT"],
    }
    messy = rng.choice(n, size=int(0.06 * n), replace=False)
    original = raw.loc[messy, "branch"].to_numpy()
    raw.loc[messy, "branch"] = [str(rng.choice(variants[name])) for name in original]

    # (c) the till was reconfigured in April 2025 and the date format changed with it
    raw["date"] = np.where(
        raw["date"] < pd.Timestamp("2025-04-01"),
        raw["date"].dt.strftime("%d/%m/%Y"),
        raw["date"].dt.strftime("%Y-%m-%d"),
    )

    # (d) the payment field is optional on the till
    raw.loc[rng.choice(n, size=int(0.03 * n), replace=False), "payment"] = None

    # (e) a handful of impossible prices
    raw.loc[rng.choice(n, size=12, replace=False), "unit_price_usd"] = 0.0
    raw.loc[rng.choice(n, size=3, replace=False), "unit_price_usd"] = 250.0

    # (f) the export was run twice for one week, so some rows appear twice
    raw = pd.concat([raw, raw.sample(n=48, random_state=7)], ignore_index=True)
    return raw.sample(frac=1.0, random_state=11).reset_index(drop=True)


branches = pd.DataFrame([
    {"branch": name, "district": info["district"], "seats": info["seats"],
     "opened": pd.Timestamp(info["opened"])}
    for name, info in BRANCHES.items()
])

raw = build_raw_extract()
print(raw.head(8).to_string(index=False))

In [ ]:
def audit(df: pd.DataFrame, name: str = "dataset") -> pd.DataFrame:
    """Questions 1 and 2 of the audit: how big, what types, what is missing."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
        "example": [df[column].dropna().iloc[0] if df[column].notna().any() else None
                    for column in df.columns],
    })
    print(report.to_string())
    return report


report = audit(raw, "raw till export")

Three things in that table should stop you.

- `date` has dtype `object` or `str`, not a datetime. It has not been parsed, and until it is,
  nothing time-based will work.
- `payment` has missing values. Are they missing at random, or concentrated somewhere?
- `branch` has more than four unique values. There are four branches.

The missingness-by-branch table below is worth looking at twice. It has thirteen rows, because
the branch column has thirteen spellings — so the grouping is already wrong, and every
`groupby("branch")` in the rest of the analysis would be wrong in the same way until the
column is cleaned.

The remaining audit questions — duplicates, categories, ranges, the time axis — need their own
checks.

In [ ]:
# Question 3 — duplicates. On the key that is supposed to be unique.
duplicate_ids = raw["transaction_id"].duplicated().sum()
print(f"duplicated transaction_id values : {duplicate_ids}")
print(f"fully identical rows             : {raw.duplicated().sum()}")

# Question 4 — are the categories what they claim to be?
print(f"\nbranch, as exported ({raw['branch'].nunique()} distinct values):")
print(raw["branch"].value_counts().to_string())

# Is `payment` missing at random, or concentrated in one branch or period?
missing_by_branch = raw.assign(missing=raw["payment"].isna()).groupby("branch")["missing"].mean()
print(f"\nshare of rows with no payment method, by exported branch label:")
print((missing_by_branch * 100).round(2).to_string())

In [ ]:
# Question 5 — are the numbers possible?
print(raw[["hour", "quantity", "unit_price_usd"]].describe().round(3).to_string())

print(f"\nnegative quantities : {(raw['quantity'] < 0).sum()}  (refunds, not errors)")
print(f"zero prices         : {(raw['unit_price_usd'] <= 0).sum()}")
print(f"prices above $20    : {(raw['unit_price_usd'] > 20).sum()}")
print("\nThe menu tops out at $3.50, and $3.85 after a price change. A $250 coffee is a")
print("keying error; a $0.00 coffee is a till fault. Neither is a data point.")

Question 6 — the time axis — needs the dates parsed first, and parsing them is the first real
decision of the project.

---

## 2.3 Clean

Two formats appear in the same column because the till was reconfigured in April 2025. The
one-liner `pd.to_datetime(raw["date"])` would either fail or, worse, succeed on some rows and
silently misread others.

**The ambiguity is not hypothetical.** `04/03/2025` is 4 March in Cambodia and 3 April in the
United States. Guessing wrong moves every date whose day is 12 or less, which is 40% of the
data, and nothing in the output will look wrong.

The safe method is to parse each format explicitly and combine.

In [ ]:
def parse_dates(text: pd.Series) -> pd.Series:
    """Parse a column holding two known date formats, without guessing between them."""
    iso = pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    local = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")

    both = iso.notna() & local.notna()
    if both.any():
        raise ValueError(f"{both.sum()} values parse under both formats — ambiguous")

    parsed = iso.fillna(local)
    if parsed.isna().any():
        raise ValueError(f"{parsed.isna().sum()} values match neither format")
    return parsed


dates = parse_dates(raw["date"])
print(f"parsed {len(dates):,} dates, {dates.min().date()} to {dates.max().date()}")
print(f"rows in the old dd/mm/yyyy format : {(dates < pd.Timestamp('2025-04-01')).sum():,}")
print("\nThe guard matters: if any value parsed under BOTH formats the function refuses,")
print("rather than picking one. That is the failure this code exists to prevent.")

In [ ]:
def clean_transactions(raw: pd.DataFrame) -> pd.DataFrame:
    """Turn the till export into an analysable table. Every step is recorded and printed.

    Returns
    -------
    pandas.DataFrame
        A copy. The input is not modified.
    """
    df = raw.copy()
    steps = []
    start_rows = len(df)

    # 1. the export ran twice for one week
    df = df.drop_duplicates(subset="transaction_id", keep="first")
    steps.append(("duplicate transaction ids removed", start_rows - len(df)))

    # 2. branch names were typed by hand
    before_names = df["branch"].nunique()
    df["branch"] = (df["branch"].astype("string")
                    .str.strip()
                    .str.replace(r"\s+", " ", regex=True)
                    .str.title())
    steps.append(("branch spellings collapsed", before_names - df["branch"].nunique()))

    # 3. dates
    df["date"] = parse_dates(df["date"])

    # 4. refunds are real transactions, not errors. Label them; do not delete them.
    df["is_refund"] = df["quantity"] < 0
    steps.append(("refund lines labelled", int(df["is_refund"].sum())))

    # 5. impossible prices are removed, and the count is reported
    impossible = (df["unit_price_usd"] <= 0) | (df["unit_price_usd"] > 20)
    df = df.loc[~impossible]
    steps.append(("impossible prices removed", int(impossible.sum())))

    # 6. derived columns
    df["line_total_usd"] = (df["quantity"] * df["unit_price_usd"]).round(2)
    df["sale_units"] = np.where(df["is_refund"], 0, df["quantity"])

    print(f"{start_rows:,} rows in")
    for label, count in steps:
        print(f"   {label:<38} {count:>6,}")
    print(f"{len(df):,} rows out\n")
    print(f"branches after cleaning: {sorted(df['branch'].unique())}")
    return df.reset_index(drop=True)


clean = clean_transactions(raw)

Note what the cleaning function does **not** do.

- It does not delete the refunds. A refund is a real event; removing it would overstate
  revenue. It labels them so that later steps can choose.
- It does not fill the missing `payment` values. Nothing in this analysis uses that column, and
  inventing values for a column you are not going to use is pure risk.
- It does not edit the input. `raw` is still the export as received, which means the cleaning
  can be re-run when next month's file arrives.

It also **prints what it did**. A cleaning step that silently drops 4,000 rows is how an
analysis goes wrong without anybody noticing.

---

## 2.4 Build the daily panel

In [ ]:
def build_daily_panel(clean: pd.DataFrame, branches: pd.DataFrame) -> pd.DataFrame:
    """One row per branch per day, with the branch metadata joined on."""
    panel = (clean.groupby(["branch", "date"], as_index=False)
             .agg(revenue_usd=("line_total_usd", "sum"),
                  units_sold=("sale_units", "sum"),
                  lines=("transaction_id", "size"),
                  refunds=("is_refund", "sum")))

    panel = panel.merge(branches, on="branch", how="left", validate="many_to_one")
    if panel["district"].isna().any():
        raise ValueError("a branch in the transactions has no metadata row")

    panel["weekday"] = panel["date"].dt.day_name()
    panel["in_term"] = panel["date"].map(in_term)
    panel["after_launch"] = panel["date"] >= LOYALTY_LAUNCH
    return panel.sort_values(["branch", "date"]).reset_index(drop=True)


panel = build_daily_panel(clean, branches)
print(panel.head().to_string(index=False))

expected_days = (END - START).days + 1
print(f"\nexpected days per branch : {expected_days}")
print(panel.groupby("branch")["date"].agg(["count", "min", "max"]).to_string())
print("\nThree branches have a row for every one of the 455 days. The Airport branch does not.")
print("A day with no transactions produces no row at all, so the gap does not appear as a")
print("missing value anywhere — it appears as two rows that are simply absent. That is the")
print("sixth audit question, and counting is how you find it.")

In [ ]:
# Question 6, properly: plot the daily transaction count and look at it.
fig, ax = plt.subplots(figsize=(11, 4))
for colour, branch in zip(PALETTE, sorted(panel["branch"].unique())):
    subset = panel.loc[panel["branch"] == branch]
    ax.plot(subset["date"], subset["lines"], lw=0.9, alpha=0.85, color=colour, label=branch)

ax.axvspan(RENOVATION_START, RENOVATION_END, color=PALETTE[3], alpha=0.18)
ax.annotate("Airport closed for refit", xy=(RENOVATION_START, 8),
            xytext=(pd.Timestamp("2025-04-20"), 12), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="0.3"))
ax.axvline(LOYALTY_LAUNCH, color="black", ls="--", lw=1.2)
ax.text(LOYALTY_LAUNCH, ax.get_ylim()[1] * 0.97, "  loyalty launch", fontsize=9, va="top")

ax.set_title("Daily transactions by branch — look before you compute anything", loc="left")
ax.set_ylabel("transactions per day")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()
plt.show()

print("The Airport branch collapses for three weeks in August. Nothing in the audit table")
print("said so. One plot of the time axis did.")

That dip is the whole reason audit question 6 exists. It is not missing data, it is not an
outlier, and no summary statistic would have shown it. The Airport branch was closed for a
refit from 10 to 31 August 2025, running one till.

Remember it. It becomes important in section 2.7.

---

## 2.5 Explore — answering Q1 and Q2

In [ ]:
# Q1 — how do the branches compare?
summary = (panel.groupby("branch")
           .agg(days=("date", "size"),
                revenue_per_day=("revenue_usd", "mean"),
                units_per_day=("units_sold", "mean"),
                refund_rate=("refunds", "sum"))
           .join(branches.set_index("branch")[["district", "seats"]]))
summary["refund_rate"] = (summary["refund_rate"] / panel.groupby("branch")["lines"].sum() * 100)
summary["revenue_per_seat"] = summary["revenue_per_day"] / summary["seats"]
summary = summary.sort_values("revenue_per_day", ascending=False)
print(summary.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
order = summary.index.tolist()

# (a) revenue per day — the obvious ranking
axes[0].barh(order[::-1], summary.loc[order[::-1], "revenue_per_day"], color=PALETTE[0])
axes[0].set_title("(a) mean revenue per day", loc="left")
axes[0].set_xlabel("USD")
axes[0].set_xlim(0)

# (b) revenue per seat — a different ranking, and a fairer one
per_seat = summary.sort_values("revenue_per_seat")
axes[1].barh(per_seat.index, per_seat["revenue_per_seat"], color=PALETTE[1])
axes[1].set_title("(b) revenue per seat per day", loc="left")
axes[1].set_xlabel("USD per seat")
axes[1].set_xlim(0)

# (c) the weekly pattern, which differs by branch
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly = (panel.groupby(["branch", "weekday"])["units_sold"].mean().unstack()[weekday_order])
weekly = weekly.div(weekly.mean(axis=1), axis=0) * 100
for colour, branch in zip(PALETTE, order):
    axes[2].plot(range(7), weekly.loc[branch], "o-", color=colour, label=branch, ms=4)
axes[2].axhline(100, color="0.6", lw=1, ls="--")
axes[2].set_xticks(range(7), [day[:3] for day in weekday_order])
axes[2].set_title("(c) weekly pattern, each branch indexed to its own mean", loc="left")
axes[2].set_ylabel("percent of branch mean")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

Panels (a) and (b) rank the branches differently, and the difference is the finding. Riverside
takes the most money; per seat it is not the most productive. Which ranking is right depends on
the decision — (a) for "where does the money come from", (b) for "where is capacity being
wasted".

Panel (c) answers Q2 and is directly actionable. Riverside peaks at the weekend; Toul Kork and
Sen Sok collapse at the weekend and peak midweek. They should not have the same staffing
roster, and indexing each branch to its own mean is what makes that visible on one axis.

In [ ]:
# The term effect at Toul Kork, with an interval rather than an impression.
toul = panel.loc[panel["branch"] == "Toul Kork"]
in_term_units = toul.loc[toul["in_term"], "units_sold"]
out_term_units = toul.loc[~toul["in_term"], "units_sold"]

test = stats.ttest_ind(in_term_units, out_term_units, equal_var=False)
interval = test.confidence_interval(0.95)
ratio = in_term_units.mean() / out_term_units.mean()

print("Toul Kork, units sold per day")
print(f"  in term      {in_term_units.mean():7.2f}   (n = {in_term_units.size})")
print(f"  out of term  {out_term_units.mean():7.2f}   (n = {out_term_units.size})")
print(f"  difference   {in_term_units.mean() - out_term_units.mean():7.2f} units per day")
print(f"  95% CI       [{interval.low:.2f}, {interval.high:.2f}]")
print(f"  ratio        {ratio:.3f}  ->  {100 * (ratio - 1):+.1f}%")
print(f"  p            {test.pvalue:.2e}")

# The same comparison at a branch with no campus, as a check that it is the campus.
for other in ["Riverside", "Sen Sok", "Airport"]:
    subset = panel.loc[panel["branch"] == other]
    other_ratio = (subset.loc[subset["in_term"], "units_sold"].mean()
                   / subset.loc[~subset["in_term"], "units_sold"].mean())
    print(f"  same calculation at {other:<10} {100 * (other_ratio - 1):+.1f}%")

That is a finding in the form section 1.7 described: a quantity, a size, an interval, a
comparison, and a use. The last four lines are the comparison — the effect is large at the
campus branch and small everywhere else, which is what makes "the campus" the explanation
rather than "the season".

Now Q3, which is the one the owner is paying for.

---

## 2.6 The headline answer, and why it is wrong

The obvious calculation. Take the three loyalty branches, compare mean daily revenue in the
quarter before the launch with the quarter after it.

In [ ]:
BEFORE = (pd.Timestamp("2025-07-01"), pd.Timestamp("2025-09-30"))
AFTER = (pd.Timestamp("2025-10-01"), pd.Timestamp("2025-12-31"))


def window(panel, start, end, branches=None):
    """Rows within a date window, optionally restricted to some branches."""
    mask = panel["date"].between(start, end)
    if branches is not None:
        mask &= panel["branch"].isin(branches)
    return panel.loc[mask]


naive_before = window(panel, *BEFORE, LOYALTY_BRANCHES)["revenue_usd"].mean()
naive_after = window(panel, *AFTER, LOYALTY_BRANCHES)["revenue_usd"].mean()
naive_change = naive_after / naive_before - 1

print("Loyalty branches, mean revenue per day")
print(f"  Jul-Sep 2025 (before)  ${naive_before:7.2f}")
print(f"  Oct-Dec 2025 (after)   ${naive_after:7.2f}")
print(f"  change                 {100 * naive_change:+6.1f}%")
print("\nThat is the number that would go in the email.")

**Stop here.** Before believing it, the question from section 1.4: *what else changed at the
same time?*

Three things did, and each of them is in the data or in the brief.

In [ ]:
# Confounder 1 — the price of everything went up before the launch.
price_check = (clean.assign(month=clean["date"].dt.to_period("M"))
               .loc[~clean["is_refund"]]
               .groupby(["month", "item"])["unit_price_usd"].mean()
               .unstack())
print("Mean unit price by month (USD)")
print(price_check.loc["2025-07":"2025-11"].round(3).to_string())
print(f"\nPrices rose on {PRICE_RISE.date()} — two weeks BEFORE the loyalty launch.")
print("Revenue is price times volume. Most of the 'after' period is at the new prices and")
print("most of the 'before' period is at the old ones, so the comparison contains the rise.")

In [ ]:
# Confounder 2 — the control branch, which got no loyalty programme, moved too.
control_before_all = window(panel, *BEFORE, CONTROL_BRANCH)["revenue_usd"].mean()
control_after = window(panel, *AFTER, CONTROL_BRANCH)["revenue_usd"].mean()

print("Airport branch — no loyalty programme at all")
print(f"  Jul-Sep 2025  ${control_before_all:7.2f}")
print(f"  Oct-Dec 2025  ${control_after:7.2f}")
print(f"  change        {100 * (control_after / control_before_all - 1):+6.1f}%")
print("\n+42% is not credible for a branch that changed nothing, so the 'before' figure is")
print("wrong. The chart in section 2.4 already showed why.")

# Confounder 3 — the refit days are real, and they are not normal trading.
refit_days = window(panel, RENOVATION_START, RENOVATION_END, CONTROL_BRANCH)
before_rows = window(panel, *BEFORE, CONTROL_BRANCH)
control_before = before_rows.loc[
    ~before_rows["date"].between(RENOVATION_START, RENOVATION_END), "revenue_usd"].mean()
control_change = control_after / control_before - 1

print(f"\n  {len(refit_days)} of the control's {len(before_rows)} 'before' days are refit days,")
print(f"  averaging ${refit_days['revenue_usd'].mean():.2f} against ${control_before:.2f} on "
      "its normal days.")
print("  They drag the 'before' mean down, which inflates the apparent growth.")

print(f"\n  control branch, refit days excluded   {100 * control_change:+6.1f}%")
print(f"  loyalty branches                      {100 * naive_change:+6.1f}%")
print("\nThe branch that never had the programme grew by very nearly the same amount as the")
print("three branches that did.")

Three problems, and each one has a fix.

| Problem | Why it breaks the comparison | Fix |
|---|---|---|
| Prices rose on 15 September | Revenue mixes price with volume | Measure **units sold**, not revenue |
| The Airport refit falls in the "before" window | The control's baseline is not its normal trading | **Exclude** 10–31 August from every window |
| Everything else moved too | Before-and-after has no counterfactual | Use the **Airport branch as a control** |

Once the control's baseline is repaired, the headline evaporates on its own: the loyalty
branches grew about 15% and the branch with no programme grew about 14%. Whatever caused the
increase, it was not the loyalty programme.

**One point about this notebook, before the corrected analysis.** All three facts were visible
here: the price rise is in the generator, the refit is in the chart, the control is in the
brief. In a real project none of them arrives on its own. The price rise is something the
manager remembers. The refit is something you notice because you plotted the time axis. The
control exists because somebody wrote down which branches were included.

**Ask what else changed. Then ask again.**

---

## 2.7 The honest answer

In [ ]:
# Exclude the refit window everywhere, so every branch is compared over the same days.
usable = panel.loc[~panel["date"].between(RENOVATION_START, RENOVATION_END)]
print(f"{len(panel) - len(usable)} branch-days removed (10-31 August, all branches)")


def did_on_logs(panel, value_col, treated, control, before, after):
    """Difference-in-differences on log(value): the result is a proportional effect.

    Returns
    -------
    (estimate, cells) : float and dict
        `estimate` is the log-scale difference of differences.
        `cells` holds the four mean log values it was built from.
    """
    cells = {}
    for label, group in [("treated", treated), ("control", control)]:
        for period, (start, end) in [("before", before), ("after", after)]:
            values = window(panel, start, end, group)[value_col].to_numpy(dtype=float)
            if values.size == 0:
                raise ValueError(f"no rows for {label} {period}")
            if (values <= 0).any():
                raise ValueError(f"{label} {period} contains non-positive values")
            cells[(label, period)] = float(np.log(values).mean())

    treated_change = cells[("treated", "after")] - cells[("treated", "before")]
    control_change = cells[("control", "after")] - cells[("control", "before")]
    return treated_change - control_change, cells


estimate, cells = did_on_logs(usable, "units_sold", LOYALTY_BRANCHES, CONTROL_BRANCH,
                              BEFORE, AFTER)

for label in ["treated", "control"]:
    change = cells[(label, "after")] - cells[(label, "before")]
    print(f"  {label:<8} units per day  {np.exp(cells[(label, 'before')]):7.2f}  ->  "
          f"{np.exp(cells[(label, 'after')]):7.2f}   {100 * (np.exp(change) - 1):+6.2f}%")

print(f"\n  difference-in-differences  {100 * (np.exp(estimate) - 1):+.2f}%")

In [ ]:
def bootstrap_did(panel, value_col, treated, control, before, after, n_boot=2000, seed=5):
    """Percentile bootstrap for the DiD estimate, resampling days within each cell."""
    cells = {}
    for label, group in [("treated", treated), ("control", control)]:
        for period, (start, end) in [("before", before), ("after", after)]:
            cells[(label, period)] = np.log(
                window(panel, start, end, group)[value_col].to_numpy(dtype=float))

    boot_rng = np.random.default_rng(seed)
    estimates = np.empty(n_boot)
    for i in range(n_boot):
        means = {}
        for key, values in cells.items():
            index = boot_rng.integers(0, values.size, values.size)
            means[key] = values[index].mean()
        estimates[i] = ((means[("treated", "after")] - means[("treated", "before")])
                        - (means[("control", "after")] - means[("control", "before")]))
    return estimates


draws = bootstrap_did(usable, "units_sold", LOYALTY_BRANCHES, CONTROL_BRANCH, BEFORE, AFTER)
low, high = np.percentile(draws, [2.5, 97.5])

print(f"effect on units sold   {100 * (np.exp(estimate) - 1):+.2f}%")
print(f"95% CI                 [{100 * (np.exp(low) - 1):+.2f}%, "
      f"{100 * (np.exp(high) - 1):+.2f}%]")
print(f"\nHeadline from section 2.6  {100 * naive_change:+.1f}%  (revenue, no control)")

This notebook can do something no real project can: check the answer.

The generator in section 2.2 applies a **3% lift** to the loyalty branches after launch. That is
the truth. Our corrected estimate is close to zero, with an interval ten points wide in each
direction — so the estimate is consistent with the truth, and also consistent with the programme
doing nothing, and also consistent with it doing harm.

The estimator is not broken. The **study** is too small, and that is a separate problem with a
separate fix.

In [ ]:
TRUE_LIFT = 0.03          # known only because we generated the data

spread = draws.std(ddof=1)
needed_spread = np.log(1 + TRUE_LIFT) / stats.norm.ppf(0.975)
data_multiple = (spread / needed_spread) ** 2

print(f"true effect applied by the generator   {100 * TRUE_LIFT:+.1f}%")
print(f"our estimate                           {100 * (np.exp(estimate) - 1):+.2f}%")
print(f"95% CI                                 [{100 * (np.exp(low) - 1):+.2f}%, "
      f"{100 * (np.exp(high) - 1):+.2f}%]")
print(f"\nstandard error of the estimate, in logs   {spread:.4f}")
print(f"standard error needed to detect {100 * TRUE_LIFT:.0f}%       {needed_spread:.4f}")
print(f"data needed, as a multiple of what we have   {data_multiple:.1f}x")
print(f"\nOne quarter of four branches cannot measure a 3% effect. Roughly "
      f"{data_multiple:.0f} times as\nmuch data would be needed — which in practice means "
      "more branches, not more months.")
print("\nThis is Week 12's lesson in a real setting: a result that fails to reach significance")
print("is not evidence of no effect. Here it is evidence of a study that could not have")
print("detected the effect that was actually there.")

In [ ]:
# The assumption behind the estimator, checked rather than assumed.
weekly_units = (usable.assign(week=usable["date"].dt.to_period("W").dt.start_time,
                              side=np.where(usable["branch"].isin(LOYALTY_BRANCHES),
                                            "loyalty branches", "control"))
                .groupby(["side", "week"])["units_sold"].mean().unstack(0))
indexed = weekly_units / weekly_units.loc[:LOYALTY_LAUNCH].mean() * 100

fig, ax = plt.subplots(figsize=(11, 4))
for colour, column in zip([PALETTE[0], PALETTE[3]], indexed.columns):
    ax.plot(indexed.index, indexed[column], lw=1.4, color=colour, label=column)
ax.axvline(LOYALTY_LAUNCH, color="black", ls="--", lw=1.2)
ax.axvline(PRICE_RISE, color=PALETTE[2], ls=":", lw=1.2)
ax.text(LOYALTY_LAUNCH, indexed.max().max(), "  loyalty launch", fontsize=9, va="top")
ax.text(PRICE_RISE, indexed.min().min(), "price rise  ", fontsize=9, ha="right", va="bottom")
ax.axhline(100, color="0.7", lw=1)
ax.set_title("Parallel trends check — weekly units, each series indexed to its own "
             "pre-launch mean", loc="left")
ax.set_ylabel("index, pre-launch mean = 100")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# A placebo: run the same estimator on a window where nothing was launched.
placebo_before = (pd.Timestamp("2025-02-01"), pd.Timestamp("2025-04-30"))
placebo_after = (pd.Timestamp("2025-05-01"), pd.Timestamp("2025-07-31"))
placebo, _ = did_on_logs(usable, "units_sold", LOYALTY_BRANCHES, CONTROL_BRANCH,
                         placebo_before, placebo_after)
print(f"placebo DiD, Feb-Apr vs May-Jul 2025 (no programme): "
      f"{100 * (np.exp(placebo) - 1):+.2f}%")
print("\nIf the assumption held perfectly the placebo would be zero. It is not, because the")
print("university term ends in May and only one of these branches is beside a campus.")
print("That is a reason to treat the confidence interval as a lower bound on the real")
print("uncertainty, not a reason to discard the estimate.")

---

## 2.8 Communicating the result

One figure and five sentences. Everything above was to earn the right to write them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

# (a) the two answers, side by side
labels = ["headline\n(revenue, no control)", "corrected\n(units, vs control)"]
values = [100 * naive_change, 100 * (np.exp(estimate) - 1)]
bars = axes[0].bar(labels, values, color=[PALETTE[3], PALETTE[2]], width=0.55)
axes[0].errorbar(1, values[1],
                 yerr=[[values[1] - 100 * (np.exp(low) - 1)],
                       [100 * (np.exp(high) - 1) - values[1]]],
                 fmt="none", ecolor="black", capsize=6, lw=1.5)
axes[0].axhline(0, color="black", lw=1)
for bar, value in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value + 0.6, f"{value:+.1f}%",
                 ha="center", fontsize=10, fontweight="bold")
axes[0].set_ylabel("change after launch")
axes[0].set_title("(a) the same programme, measured two ways", loc="left")
axes[0].set_ylim(min(0, 100 * (np.exp(low) - 1)) - 3, max(values) + 4)

# (b) what the corrected estimate is built from
monthly = (usable.assign(month=usable["date"].dt.to_period("M").dt.start_time,
                         side=np.where(usable["branch"].isin(LOYALTY_BRANCHES),
                                       "loyalty branches", "control"))
           .groupby(["side", "month"])["units_sold"].mean().unstack(0))
monthly = monthly / monthly.loc[:LOYALTY_LAUNCH].mean() * 100
for colour, column in zip([PALETTE[0], PALETTE[3]], monthly.columns):
    axes[1].plot(monthly.index, monthly[column], "o-", ms=4, lw=1.6, color=colour, label=column)
axes[1].axvline(LOYALTY_LAUNCH, color="black", ls="--", lw=1.2)
axes[1].axhline(100, color="0.7", lw=1)
axes[1].set_title("(b) monthly units, indexed to each series' pre-launch mean", loc="left")
axes[1].set_ylabel("index, pre-launch mean = 100")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
MONTHLY_COST = 450.0      # what the loyalty provider charges
GROSS_MARGIN = 0.62       # a plausible cafe figure; state assumptions, do not bury them

baseline_units = np.exp(cells[("treated", "before")]) * len(LOYALTY_BRANCHES)
mean_price = clean.loc[~clean["is_refund"], "unit_price_usd"].mean()


def monthly_margin(effect_in_logs):
    """Extra gross margin per month implied by a log-scale effect."""
    return baseline_units * (np.exp(effect_in_logs) - 1) * mean_price * GROSS_MARGIN * 30


print("FINDINGS — Bopha Coffee loyalty programme, Oct-Dec 2025")
print("=" * 74)
print(f"""
1. Units sold per day at the three loyalty branches moved {100 * (np.exp(estimate) - 1):+.1f}% relative to the
   Airport branch after launch (95% CI {100 * (np.exp(low) - 1):+.1f}% to {100 * (np.exp(high) - 1):+.1f}%). The data cannot
   distinguish that from no effect in either direction.

2. The {100 * naive_change:+.0f}% obtained by comparing revenue before and after is not the programme.
   Prices rose on {PRICE_RISE.strftime('%d %B')}, two weeks before launch, and the control branch
   - which never had the programme - grew {100 * control_change:+.0f}% over the same quarter once its
   refit fortnight is excluded.

3. Even at the most favourable end of the interval the programme is marginal: a
   {100 * (np.exp(high) - 1):+.1f}% lift is about ${monthly_margin(high):,.0f} a month in extra gross margin against a cost
   of ${MONTHLY_COST:,.0f}. At the point estimate it loses money.

4. Assumptions: units are net of refunds; 10-31 August is excluded because the Airport
   branch was closed for a refit; gross margin is taken as {GROSS_MARGIN:.0%}; mean unit price
   ${mean_price:.2f}. A placebo test on an untreated period returns {100 * (np.exp(placebo) - 1):+.1f}%, so the real
   uncertainty is wider than the interval above.

5. Recommendation: do not renew on this evidence - and do not conclude that the programme
   failed either, because this study could not have detected a 3% effect. Roll it out to
   the Airport branch on a fixed date, hold one branch back for a further two quarters,
   and record the launch date and every price change. That design would answer the
   question.
""")
print("=" * 74)

Read that against the five parts in section 1.7: what was measured, how much, how certain,
compared with what, and so what. Every sentence carries its own uncertainty, the assumptions
are stated rather than buried, and the recommendation is specific enough to act on.

Note what is absent. There is no claim that the programme failed — the data do not support that
either. "We cannot tell, and here is the design that would" is a legitimate and often correct
finding, and it is far more useful than a confident number built on a broken comparison.

---

---

# Part 3 — Guided workshop (in class)

Four tasks. They follow the same order the case study did: audit, clean, aggregate, compare.

### Task 1 — Audit a second extract

A colleague sends twelve rows from another till. Print it, look at it, and then count the
faults.

**Problem.** Assign four variables from `second_extract`:

| Variable | Quantity |
|---|---|
| `n_duplicate_ids` | how many rows have a `transaction_id` seen earlier in the frame |
| `n_missing_payment` | how many rows have no payment method |
| `n_branch_labels` | how many distinct branch strings appear, **before** cleaning |
| `n_impossible_prices` | how many rows have a price of zero or above $20 |

**Hints**

- `Series.duplicated()` marks the *later* occurrences; `.sum()` counts them.
- `Series.isna().sum()` for the missing count.
- `Series.nunique()` for the label count.
- Combine two conditions with `|`, and remember the brackets.

**Expected output.** Four small integers, then `Task 1 complete`.

In [ ]:
second_extract = pd.DataFrame({
    "transaction_id": ["X001", "X002", "X003", "X004", "X005", "X006",
                       "X007", "X008", "X003", "X009", "X010", "X011"],
    "branch": ["Riverside", "riverside", "Sen Sok", "SEN SOK", "Sen  Sok", "Airport",
               "airport", "Riverside", "Sen Sok", "Airport", "Riverside", "Riverside"],
    "date": ["2026-02-01", "2026-02-01", "2026-02-02", "2026-02-02", "2026-02-03",
             "2026-02-03", "2026-02-04", "2026-02-04", "2026-02-02", "2026-02-05",
             "2026-02-05", "2026-02-06"],
    "item": ["Iced coffee", "Hot coffee", "Tea", "Iced latte", "Pastry", "Sandwich",
             "Iced coffee", "Tea", "Tea", "Bottled water", "Iced coffee", "Iced latte"],
    "quantity": [1, 2, 1, -1, 3, 1, 1, 2, 1, 1, 1, 2],
    "unit_price_usd": [1.75, 1.50, 0.00, 2.25, 1.60, 3.50,
                       250.00, 1.25, 0.00, 0.75, 1.75, 2.25],
    "payment": ["Cash", "QR", None, "Card", "Cash", None,
                "QR", "Cash", None, "Card", "QR", "Cash"],
})
print(second_extract.to_string(index=False))

In [ ]:
# TODO 1: count the four faults in second_extract.
n_duplicate_ids = None
n_missing_payment = None
n_branch_labels = None
n_impossible_prices = None

if None not in (n_duplicate_ids, n_missing_payment, n_branch_labels, n_impossible_prices):
    print(f"duplicate transaction ids : {n_duplicate_ids}")
    print(f"missing payment methods   : {n_missing_payment}")
    print(f"distinct branch labels    : {n_branch_labels}")
    print(f"impossible prices         : {n_impossible_prices}")

In [ ]:
# Verification for Task 1.
truth = {
    "n_duplicate_ids": int(second_extract["transaction_id"].duplicated().sum()),
    "n_missing_payment": int(second_extract["payment"].isna().sum()),
    "n_branch_labels": int(second_extract["branch"].nunique()),
    "n_impossible_prices": int(((second_extract["unit_price_usd"] <= 0)
                                | (second_extract["unit_price_usd"] > 20)).sum()),
}
given = {
    "n_duplicate_ids": n_duplicate_ids,
    "n_missing_payment": n_missing_payment,
    "n_branch_labels": n_branch_labels,
    "n_impossible_prices": n_impossible_prices,
}
problems = []
for key, expected in truth.items():
    value = given[key]
    if value is None:
        problems.append(f"{key} is still None")
    elif int(value) != expected:
        hint = ""
        if key == "n_duplicate_ids" and int(value) == expected * 2:
            hint = " — duplicated(keep=False) marks BOTH copies; the default marks the later one"
        if key == "n_branch_labels" and int(value) == 3:
            hint = " — count the labels as exported, before any cleaning"
        problems.append(f"{key} should be {expected}, got {value}{hint}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print("Task 1 complete. Twelve rows, four separate faults — and this is a clean extract")
    print("by the standards of most till systems.")

### Task 2 — Clean the branch column

**Problem.** Build `fixed_branches`: `second_extract["branch"]` normalised so that it contains
exactly **three** distinct values — `Airport`, `Riverside`, `Sen Sok` — with the same length and
the same row order as the original.

**Hints**

- Three faults are present: leading or trailing spaces, inconsistent case, and a doubled
  internal space.
- `.str.strip()`, `.str.replace(r"\s+", " ", regex=True)`, `.str.title()` — in that order.
- `.str.title()` capitalises the first letter of every word, so `"SEN SOK"` becomes `"Sen Sok"`.

**Expected output.** The three cleaned names and their counts, then `Task 2 complete`.

In [ ]:
# TODO 2: normalise the branch labels down to three.
fixed_branches = None

if fixed_branches is not None:
    print(fixed_branches.value_counts().to_string())

In [ ]:
# Verification for Task 2.
problems = []
if fixed_branches is None:
    problems.append("fixed_branches is still None")
elif not isinstance(fixed_branches, pd.Series):
    problems.append(f"expected a Series, got {type(fixed_branches).__name__}")
else:
    if len(fixed_branches) != len(second_extract):
        problems.append(f"expected {len(second_extract)} rows, got {len(fixed_branches)}")
    found = set(fixed_branches.dropna().unique())
    expected_names = {"Airport", "Riverside", "Sen Sok"}
    if found != expected_names:
        extra = sorted(found - expected_names)
        missing = sorted(expected_names - found)
        detail = []
        if extra:
            detail.append(f"unexpected: {extra}")
        if missing:
            detail.append(f"missing: {missing}")
        problems.append("the cleaned labels are wrong — " + "; ".join(detail))
    elif not fixed_branches.astype(str).str.title().equals(fixed_branches.astype(str)):
        problems.append("the values are not in Title Case")
    else:
        # row order must be preserved, so row 0 must still be a Riverside row
        if str(fixed_branches.iloc[0]) != "Riverside" or str(fixed_branches.iloc[2]) != "Sen Sok":
            problems.append("the row order changed — clean in place, do not sort or drop")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 2 complete. {second_extract['branch'].nunique()} labels became "
          f"{fixed_branches.nunique()}.")

### Task 3 — The busiest hour, per branch

Back to the main dataset. This answers a staffing question the owner did not ask but will.

**Problem.** From `clean`, build `hourly`: a DataFrame of **total units sold** with one row per
branch and one column per hour. Then assign `busiest_hour`, a dictionary mapping each branch
name to the hour with the most units.

**Hints**

- `clean.groupby(["branch", "hour"])["sale_units"].sum().unstack()` gives the table.
- `DataFrame.idxmax(axis=1)` gives the column label of the maximum in each row.
- `.to_dict()` turns the resulting Series into a dictionary.

**Expected output.** The table, the dictionary, then `Task 3 complete`.

In [ ]:
# TODO 3: units by branch and hour, then the busiest hour for each branch.
hourly = None
busiest_hour = None

if hourly is not None and busiest_hour is not None:
    print(hourly.to_string())
    print()
    for name, hour in busiest_hour.items():
        print(f"  {name:<12} busiest at {hour:02d}:00")

In [ ]:
# Verification for Task 3.
reference_hourly = clean.groupby(["branch", "hour"])["sale_units"].sum().unstack()
reference_busiest = reference_hourly.idxmax(axis=1).to_dict()
problems = []

if hourly is None or busiest_hour is None:
    problems.append("hourly or busiest_hour is still None")
else:
    if not isinstance(hourly, pd.DataFrame):
        problems.append(f"hourly should be a DataFrame, got {type(hourly).__name__}")
    elif hourly.shape != reference_hourly.shape:
        problems.append(f"hourly should be {reference_hourly.shape}, got {hourly.shape} — "
                        "one row per branch, one column per hour")
    elif not np.allclose(hourly.to_numpy(dtype=float),
                         reference_hourly.to_numpy(dtype=float)):
        by_lines = clean.groupby(["branch", "hour"])["transaction_id"].size().unstack()
        if np.allclose(hourly.to_numpy(dtype=float), by_lines.to_numpy(dtype=float)):
            problems.append("hourly counts transactions, not units — sum sale_units")
        else:
            problems.append("hourly does not match the reference totals")

    if not isinstance(busiest_hour, dict):
        problems.append(f"busiest_hour should be a dict, got {type(busiest_hour).__name__}")
    elif {str(k) for k in busiest_hour} != set(reference_busiest):
        problems.append("busiest_hour should have one entry per branch")
    else:
        wrong = {k: v for k, v in busiest_hour.items() if int(v) != int(reference_busiest[k])}
        if wrong:
            problems.append(f"wrong hours: {wrong} — expected {reference_busiest}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 3 complete. Busiest hours: {reference_busiest}")
    peaks = sorted(set(reference_busiest.values()))
    if len(peaks) == 1:
        print(f"Every branch peaks at {peaks[0]:02d}:00. One peak across the chain means one")
        print("bottleneck, and one obvious place to add staff.")
    else:
        print(f"The peaks are not the same everywhere: {peaks}. Staffing should differ by")
        print("branch, which is the same conclusion the weekly pattern reached.")

### Task 4 — The comparison, and its control

**Problem.** Redo the section 2.6 comparison on **transactions per day** (`lines`) instead of
revenue, using `usable` so the refit days are already excluded. Assign two proportions:

| Variable | Quantity |
|---|---|
| `treated_lift` | (mean `lines` after ÷ mean `lines` before) − 1, for the three loyalty branches |
| `control_lift` | the same for the Airport branch |

Then answer, in the Markdown cell below: if you had only `treated_lift`, what would you have
concluded, and what does `control_lift` do to that conclusion?

**Hints**

- `window(usable, *BEFORE, LOYALTY_BRANCHES)` and `window(usable, *AFTER, LOYALTY_BRANCHES)`.
- `CONTROL_BRANCH` is already defined as a list.
- These are proportions, not percentages: 0.05, not 5.

**Expected output.** Two proportions, then `Task 4 complete`.

In [ ]:
# TODO 4: the lift in transactions per day, for the treated branches and for the control.
treated_lift = None
control_lift = None

if None not in (treated_lift, control_lift):
    print(f"loyalty branches  {100 * treated_lift:+6.2f}%")
    print(f"control branch    {100 * control_lift:+6.2f}%")
    print(f"difference        {100 * (treated_lift - control_lift):+6.2f} points")

In [ ]:
# Verification for Task 4.
expected_treated = (window(usable, *AFTER, LOYALTY_BRANCHES)["lines"].mean()
                    / window(usable, *BEFORE, LOYALTY_BRANCHES)["lines"].mean() - 1)
expected_control = (window(usable, *AFTER, CONTROL_BRANCH)["lines"].mean()
                    / window(usable, *BEFORE, CONTROL_BRANCH)["lines"].mean() - 1)
problems = []

if None in (treated_lift, control_lift):
    problems.append("treated_lift or control_lift is still None")
else:
    for name, value, expected in [("treated_lift", treated_lift, expected_treated),
                                  ("control_lift", control_lift, expected_control)]:
        if np.isclose(value, 100 * expected, atol=1e-6):
            problems.append(f"{name} is a percentage — the task asks for a proportion")
        elif np.isclose(value, expected + 1, atol=1e-6):
            problems.append(f"{name} is a ratio — subtract 1 to make it a lift")
        elif not np.isclose(value, expected, atol=1e-6):
            problems.append(f"{name} should be {expected:.6f}, got {value:.6f}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 4 complete. Treated {100 * treated_lift:+.2f}%, "
          f"control {100 * control_lift:+.2f}%.")
    print("The branch with no loyalty programme moved by as much as the three that had it.")
    print("Without the control you would have reported the treated figure as the effect.")

**Write your answer here.** Two or three sentences: what `treated_lift` alone would have led
you to conclude, and what `control_lift` changes.

*Replace this line with your answer.*

---

# Part 4 — Homework

Two problems. Both are tools you will use again in the final project, which is why they are
written as reusable functions rather than as one-off calculations.

---

## Problem 1 — A reusable audit

**Problem.** Write `audit_frame(df)`.

It returns a **DataFrame** with one row per column of `df` and exactly these columns, in this
order:

| Column | Contents |
|---|---|
| `column` | the column name |
| `dtype` | its dtype, as a string |
| `n_missing` | number of missing values |
| `pct_missing` | percentage missing, rounded to 2 decimal places |
| `n_unique` | number of distinct non-missing values |
| `example` | the first non-missing value, or `None` if the column is entirely missing |

**Requirements**

- Sort by `n_missing` **descending**, then by `column` **ascending**, and reset the index to
  `0, 1, 2, …`.
- Raise `TypeError` if `df` is not a DataFrame.
- Raise `ValueError` if `df` has no columns.
- Do not modify the input.

**Hints**

- `df.dtypes`, `df.isna().sum()`, `df.nunique()` each give a Series indexed by column name.
- `df[col].dropna().iloc[0]` is the first non-missing value; guard the empty case.
- `sort_values(by=[...], ascending=[False, True])` sorts on two keys with different directions.

In [ ]:
# Problem 1 — starter code.

def audit_frame(df) -> pd.DataFrame:
    """Summarise every column of a DataFrame: type, missingness, cardinality, an example.

    Parameters
    ----------
    df : pandas.DataFrame
        The frame to audit. Not modified.

    Returns
    -------
    pandas.DataFrame
        One row per column, with columns
        [column, dtype, n_missing, pct_missing, n_unique, example],
        sorted by n_missing descending then column ascending, index reset.

    Raises
    ------
    TypeError
        If df is not a DataFrame.
    ValueError
        If df has no columns.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete audit_frame()")


print(audit_frame(raw).to_string(index=False))

In [ ]:
# Problem 1 — assertions. Run until this prints "all assertions passed".

check = pd.DataFrame({
    "a": [1, 2, 3, 4],                       # no missing, 4 unique
    "b": [None, None, "x", "x"],             # 2 missing, 1 unique
    "c": [1.5, None, 2.5, 2.5],              # 1 missing, 2 unique
    "d": [None, None, None, None],           # all missing, 0 unique
})

out = audit_frame(check)

assert isinstance(out, pd.DataFrame), f"Must return a DataFrame; got {type(out).__name__}"
assert list(out.columns) == ["column", "dtype", "n_missing", "pct_missing",
                             "n_unique", "example"], (
    f"Wrong columns or wrong order: {list(out.columns)}"
)
assert len(out) == 4, f"One row per column of the input: expected 4, got {len(out)}"
assert list(out.index) == [0, 1, 2, 3], f"The index must be reset; got {list(out.index)}"

# sorted by n_missing descending, then column ascending
assert list(out["column"]) == ["d", "b", "c", "a"], (
    f"Wrong order: {list(out['column'])}. Sort by n_missing descending, then column."
)
assert list(out["n_missing"]) == [4, 2, 1, 0], f"Wrong counts: {list(out['n_missing'])}"
assert np.allclose(out["pct_missing"].to_numpy(dtype=float), [100.0, 50.0, 25.0, 0.0]), (
    f"pct_missing should be a PERCENTAGE rounded to 2dp; got {list(out['pct_missing'])}"
)
assert list(out["n_unique"]) == [0, 1, 2, 4], (
    f"n_unique counts distinct NON-missing values; got {list(out['n_unique'])}"
)

examples = out.set_index("column")["example"]
assert examples["a"] == 1, f"example for 'a' should be 1; got {examples['a']!r}"
assert examples["b"] == "x", f"example for 'b' should be 'x'; got {examples['b']!r}"
assert examples["c"] == 1.5, f"example for 'c' should be 1.5; got {examples['c']!r}"
assert examples["d"] is None or pd.isna(examples["d"]), (
    "example for an all-missing column should be None"
)

assert out["dtype"].map(type).eq(str).all(), "dtype must be reported as a string"

# --- errors ---
for bad in [check["a"], "not a frame", [1, 2, 3], None]:
    try:
        audit_frame(bad)
    except TypeError:
        pass
    else:
        raise AssertionError(f"{type(bad).__name__} input must raise TypeError")

try:
    audit_frame(pd.DataFrame())
except ValueError:
    pass
else:
    raise AssertionError("A frame with no columns must raise ValueError")

# --- the input is untouched ---
before = check.copy()
_ = audit_frame(check)
assert check.equals(before), "audit_frame must not modify its input"

# --- and it survives the real export ---
real = audit_frame(raw)
assert len(real) == raw.shape[1]
assert real.loc[real["column"] == "payment", "n_missing"].iloc[0] > 0

print("Problem 1: all assertions passed.")

---

## Problem 2 — Difference-in-differences as a function

**Problem.** Write
`did_estimate(panel, value_col, unit_col, date_col, treated_units, before, after, log=True)`.

`before` and `after` are each a `(start, end)` pair of timestamps, inclusive. Every unit **not**
in `treated_units` is the control group.

It returns a dictionary with exactly these keys:

| Key | Value |
|---|---|
| `treated_before`, `treated_after` | the treated group's mean in each window |
| `control_before`, `control_after` | the control group's mean in each window |
| `treated_change`, `control_change` | after minus before, for each group |
| `estimate` | `treated_change - control_change` |
| `pct_effect` | `exp(estimate) - 1` when `log=True`, otherwise `None` |

When `log=True`, the four means are means of `np.log(value)`, not logs of the means.

**Requirements**

- Raise `KeyError` if `value_col`, `unit_col` or `date_col` is not a column.
- Raise `TypeError` if `date_col` is not a datetime column, or if `treated_units` is a string
  (a bare string would iterate character by character).
- Raise `ValueError` if the windows overlap, if no treated unit is present, if no control unit
  is present, or if any of the four windows is empty.
- Raise `ValueError` if `log=True` and any value is not strictly positive.
- Do not modify the input.

**Hints**

- `pd.api.types.is_datetime64_any_dtype(panel[date_col])` for the type check.
- `Series.between(start, end)` is inclusive at both ends.
- The windows overlap if `before[0] <= after[1]` **and** `after[0] <= before[1]`.
- Build the four means with a loop over two groups and two windows, exactly as section 2.7 did.

In [ ]:
# Problem 2 — starter code.

def did_estimate(panel, value_col: str, unit_col: str, date_col: str,
                 treated_units, before, after, log: bool = True) -> dict:
    """Difference-in-differences for one value column, with everything else as control.

    Parameters
    ----------
    panel : pandas.DataFrame
        One row per unit per period. Not modified.
    value_col, unit_col, date_col : str
        Column names.
    treated_units : iterable
        Labels of the treated units. Everything else is the control group.
    before, after : (Timestamp, Timestamp)
        Inclusive date windows. They must not overlap.
    log : bool
        If True, average log(value) so the estimate is proportional.

    Returns
    -------
    dict
        Keys: treated_before, treated_after, control_before, control_after,
        treated_change, control_change, estimate, pct_effect.

    Raises
    ------
    KeyError, TypeError, ValueError
        As described in the problem statement.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete did_estimate()")


result = did_estimate(usable, "units_sold", "branch", "date",
                      LOYALTY_BRANCHES, BEFORE, AFTER, log=True)
for key, value in result.items():
    print(f"  {key:<16} {value:.6f}" if value is not None else f"  {key:<16} None")

In [ ]:
# Problem 2 — assertions. Run until this prints "all assertions passed".

# Designed so every number can be checked on paper:
#   treated units A and B: 100 before, 121 after
#   control unit C:        100 before, 110 after
#   log DiD = log(121/100) - log(110/100) = log(1.1)  ->  pct_effect exactly 0.10
#   plain DiD = (121 - 100) - (110 - 100) = 11
rows = []
for unit, value_before, value_after in [("A", 100.0, 121.0), ("B", 100.0, 121.0),
                                        ("C", 100.0, 110.0)]:
    for day in pd.date_range("2025-01-01", periods=10, freq="D"):
        rows.append({"shop": unit, "day": day, "sales": value_before, "note": "x"})
    for day in pd.date_range("2025-02-01", periods=10, freq="D"):
        rows.append({"shop": unit, "day": day, "sales": value_after, "note": "x"})
sample_panel = pd.DataFrame(rows)

WIN_BEFORE = (pd.Timestamp("2025-01-01"), pd.Timestamp("2025-01-10"))
WIN_AFTER = (pd.Timestamp("2025-02-01"), pd.Timestamp("2025-02-10"))

out = did_estimate(sample_panel, "sales", "shop", "day", ["A", "B"],
                   WIN_BEFORE, WIN_AFTER, log=True)

assert isinstance(out, dict), f"Must return a dict; got {type(out).__name__}"
expected_keys = {"treated_before", "treated_after", "control_before", "control_after",
                 "treated_change", "control_change", "estimate", "pct_effect"}
assert set(out) == expected_keys, (
    f"Wrong keys.\n  missing: {sorted(expected_keys - set(out))}\n"
    f"  unexpected: {sorted(set(out) - expected_keys)}"
)

assert np.isclose(out["treated_before"], np.log(100.0)), (
    f"With log=True the means are means of log(value): expected {np.log(100.0):.6f}, "
    f"got {out['treated_before']:.6f}"
)
assert np.isclose(out["treated_after"], np.log(121.0))
assert np.isclose(out["control_before"], np.log(100.0))
assert np.isclose(out["control_after"], np.log(110.0))
assert np.isclose(out["treated_change"], np.log(1.21))
assert np.isclose(out["control_change"], np.log(1.10))
assert np.isclose(out["estimate"], np.log(1.1)), (
    f"estimate = treated_change - control_change = log(1.1) = {np.log(1.1):.6f}; "
    f"got {out['estimate']:.6f}"
)
assert np.isclose(out["pct_effect"], 0.10), (
    f"pct_effect should be exactly 0.10 here; got {out['pct_effect']}"
)

# --- the plain, non-log version ---
plain = did_estimate(sample_panel, "sales", "shop", "day", ["A", "B"],
                     WIN_BEFORE, WIN_AFTER, log=False)
assert np.isclose(plain["treated_before"], 100.0)
assert np.isclose(plain["treated_after"], 121.0)
assert np.isclose(plain["estimate"], 11.0), (
    f"Without logs the DiD is (121-100) - (110-100) = 11; got {plain['estimate']}"
)
assert plain["pct_effect"] is None, "pct_effect is only defined when log=True"

# --- everything not named is the control ---
one_treated = did_estimate(sample_panel, "sales", "shop", "day", ["A"],
                           WIN_BEFORE, WIN_AFTER, log=False)
assert np.isclose(one_treated["control_after"], (121.0 + 110.0) / 2), (
    "With only A treated, the control is B AND C"
)

# --- errors ---
for bad in [("nope", "shop", "day"), ("sales", "nope", "day"), ("sales", "shop", "nope")]:
    try:
        did_estimate(sample_panel, *bad, ["A"], WIN_BEFORE, WIN_AFTER)
    except KeyError:
        pass
    else:
        raise AssertionError(f"Missing column {bad} must raise KeyError")

try:
    did_estimate(sample_panel, "sales", "shop", "note", ["A"], WIN_BEFORE, WIN_AFTER)
except TypeError:
    pass
else:
    raise AssertionError("A non-datetime date_col must raise TypeError")

try:
    did_estimate(sample_panel, "sales", "shop", "day", "A", WIN_BEFORE, WIN_AFTER)
except TypeError:
    pass
else:
    raise AssertionError("A bare string for treated_units must raise TypeError")

try:
    did_estimate(sample_panel, "sales", "shop", "day", ["A", "B", "C"],
                 WIN_BEFORE, WIN_AFTER)
except ValueError:
    pass
else:
    raise AssertionError("No control units left must raise ValueError")

try:
    did_estimate(sample_panel, "sales", "shop", "day", ["Z"], WIN_BEFORE, WIN_AFTER)
except ValueError:
    pass
else:
    raise AssertionError("No treated unit present must raise ValueError")

try:
    did_estimate(sample_panel, "sales", "shop", "day", ["A"],
                 WIN_BEFORE, (pd.Timestamp("2025-01-05"), pd.Timestamp("2025-02-10")))
except ValueError:
    pass
else:
    raise AssertionError("Overlapping windows must raise ValueError")

try:
    did_estimate(sample_panel, "sales", "shop", "day", ["A"],
                 (pd.Timestamp("2024-01-01"), pd.Timestamp("2024-01-10")), WIN_AFTER)
except ValueError:
    pass
else:
    raise AssertionError("An empty window must raise ValueError")

with_zero = sample_panel.copy()
with_zero.loc[0, "sales"] = 0.0
try:
    did_estimate(with_zero, "sales", "shop", "day", ["A"], WIN_BEFORE, WIN_AFTER, log=True)
except ValueError:
    pass
else:
    raise AssertionError("log=True with a zero value must raise ValueError")

# --- the input is untouched ---
before_copy = sample_panel.copy()
_ = did_estimate(sample_panel, "sales", "shop", "day", ["A", "B"], WIN_BEFORE, WIN_AFTER)
assert sample_panel.equals(before_copy), "did_estimate must not modify its input"

# --- and it reproduces section 2.7 ---
real = did_estimate(usable, "units_sold", "branch", "date", LOYALTY_BRANCHES,
                    BEFORE, AFTER, log=True)
assert np.isclose(real["estimate"], estimate), (
    f"Should match section 2.7's estimate of {estimate:.6f}; got {real['estimate']:.6f}"
)

print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 200–300 words.

1. Section 2.6 produced a headline of about +15% and section 2.7 reduced it to roughly zero.
   Name the three things that had to be known, and say for each one **how an analyst who did
   not build the data could have found it**.
2. The corrected estimate has a confidence interval about ten points wide in each direction,
   and the true effect built into the generator was +3%. Explain what that says about the
   study — not about the programme — and what you would change about the design.
3. Take one dataset you might use for your final project. Write down the **question** in the
   form section 1.1 describes, and name **one confounder** that would make a naive
   before-and-after comparison misleading.

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] The short answer in Task 4 is written.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 200–300 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 14

Week 14 is the final project workshop. Bring:

- a dataset, or two candidates,
- the question, written in the form section 1.1 describes,
- and one sentence on what somebody would do differently depending on the answer.

The project report follows the structure of this notebook: question, audit, cleaning, explore,
model, findings. The findings section is the one that carries the marks, and section 2.8 is the
template.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §13.1–13.5.

Pfaffelhuber, P. *Python for Data Analysis*. §10.
Available at `https://pfaffelh.github.io/python_for_data/`.

Card, D., & Krueger, A. B. (1994). Minimum wages and employment: a case study of the fast-food
industry in New Jersey and Pennsylvania. *The American Economic Review*, 84(4), 772–793.